In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Kaggle competition data path
DATA_PATH = Path("/kaggle/input/competitions/datathon-2026-round-1")

# List everything that's actually there
print("Files in competition folder:")
for f in sorted(os.listdir(DATA_PATH)):
    size_mb = os.path.getsize(DATA_PATH / f) / 1e6
    print(f"  {f:35s}  {size_mb:>8.2f} MB")

Files in competition folder:
  baseline.ipynb                           0.01 MB
  customers.csv                            7.08 MB
  geography.csv                            1.40 MB
  inventory.csv                            5.67 MB
  order_items.csv                         23.94 MB
  orders.csv                              45.96 MB
  payments.csv                            18.38 MB
  products.csv                             0.20 MB
  promotions.csv                           0.00 MB
  returns.csv                              2.28 MB
  reviews.csv                              6.79 MB
  sales.csv                                0.13 MB
  sample_submission.csv                    0.02 MB
  shipments.csv                           19.76 MB
  web_traffic.csv                          0.21 MB


In [3]:
tables = {}
csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]

for f in csv_files:
    name = f.replace('.csv', '')
    tables[name] = pd.read_csv(DATA_PATH / f, low_memory=False)
    print(f"{name:25s}  {tables[name].shape[0]:>10,} rows × {tables[name].shape[1]:>2} cols")

products                        2,412 rows ×  8 cols
sample_submission                 548 rows ×  3 cols
promotions                         50 rows × 10 cols
shipments                     566,067 rows ×  4 cols
order_items                   714,669 rows ×  7 cols
reviews                       113,551 rows ×  7 cols
inventory                      60,247 rows × 17 cols
returns                        39,939 rows ×  7 cols
sales                           3,833 rows ×  3 cols
orders                        646,945 rows ×  8 cols
geography                      39,948 rows ×  4 cols
customers                     121,930 rows ×  7 cols
payments                      646,945 rows ×  4 cols
web_traffic                     3,652 rows ×  7 cols


In [ ]:
date_columns = {
    'customers': ['signup_date'],
    'orders': ['order_date'],
    'promotions': ['start_date', 'end_date'],
    'shipments': ['ship_date', 'delivery_date'],
    'returns': ['return_date'],
    'reviews': ['review_date'],
    'sales': ['Date'],
    'inventory': ['snapshot_date'],
    'web_traffic': ['date'],
}

for tbl, cols in date_columns.items():
    if tbl in tables:
        for c in cols:
            if c in tables[tbl].columns:
                tables[tbl][c] = pd.to_datetime(tables[tbl][c], errors='coerce')

# Quick health check
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(f"  Shape: {df.shape}")
    print(f"  Nulls: {df.isnull().sum().sum():,} total")
    print(f"  Dupes: {df.duplicated().sum():,}")
    print(f"  Columns: {list(df.columns)}")

In [ ]:
print(tables['order_items'].isnull().sum())

In [ ]:
print("\nOrder status breakdown:")
print(tables['orders']['order_status'].value_counts())

print("\nAvg items per order:", 
      tables['order_items'].shape[0] / tables['orders'].shape[0])

In [ ]:
# Block 1: Hidden nulls in customers
print("=" * 50)
print("CUSTOMERS — hidden nulls check")
print("=" * 50)
for col in ['gender', 'age_group', 'acquisition_channel']:
    print(f"\n{col}:")
    print(tables['customers'][col].value_counts(dropna=False).head(10))

# Block 2: Confirm order_items null structure
print("\n" + "=" * 50)
print("ORDER_ITEMS — nulls per column")
print("=" * 50)
print(tables['order_items'].isnull().sum())

# Block 3: Date ranges
print("\n" + "=" * 50)
print("DATE RANGES")
print("=" * 50)
# Ensure dates are parsed first
for tbl, col in [('sales', 'Date'), ('web_traffic', 'date'), 
                  ('orders', 'order_date'), ('customers', 'signup_date'),
                  ('returns', 'return_date'), ('reviews', 'review_date')]:
    s = pd.to_datetime(tables[tbl][col], errors='coerce')
    print(f"{tbl:15s} {col:15s}: {s.min().date()} → {s.max().date()}  ({len(s):,} rows)")

# Block 4: Geography coverage
print("\n" + "=" * 50)
print("GEOGRAPHY")
print("=" * 50)
print("Unique zips in customers:", tables['customers']['zip'].nunique())
print("Unique zips in geography:", tables['geography']['zip'].nunique())
print("Unique regions:", tables['geography']['region'].nunique())
print("Regions:", tables['geography']['region'].unique())
print("\nCustomers per region:")
cust_geo = tables['customers'].merge(tables['geography'], on='zip', how='left')
print(cust_geo['region'].value_counts(dropna=False))

In [ ]:
# ─────────────────────────────────────────────────
# MASTER JOINED TABLE — Join most tables
# ─────────────────────────────────────────────────
# Start from order_items (most granular), join everything useful

# Re-parse dates first (cleanly, in one place)
tables['orders']['order_date'] = pd.to_datetime(tables['orders']['order_date'])
tables['customers']['signup_date'] = pd.to_datetime(tables['customers']['signup_date'])
tables['returns']['return_date'] = pd.to_datetime(tables['returns']['return_date'])
tables['reviews']['review_date'] = pd.to_datetime(tables['reviews']['review_date'])
tables['sales']['Date'] = pd.to_datetime(tables['sales']['Date'])
tables['web_traffic']['date'] = pd.to_datetime(tables['web_traffic']['date'])

# Build line-item level master table
master = (
    tables['order_items']
    .merge(tables['orders'], on='order_id', how='left')
    .merge(tables['products'], on='product_id', how='left', suffixes=('', '_prod'))
    .merge(tables['customers'][['customer_id', 'zip', 'signup_date', 
                                 'gender', 'age_group', 'acquisition_channel']],
           on='customer_id', how='left', suffixes=('', '_cust'))
    .merge(tables['geography'][['zip', 'region', 'district']], 
           on='zip', how='left', suffixes=('', '_geo'))
)

# Add derived columns
master['gross_revenue'] = master['quantity'] * master['unit_price']
master['net_revenue'] = master['gross_revenue'] - master['discount_amount']
master['gross_cogs'] = master['quantity'] * master['cogs']
master['gross_profit'] = master['net_revenue'] - master['gross_cogs']
master['gross_margin'] = master['gross_profit'] / master['net_revenue']
master['has_promo'] = master['promo_id'].notna()
master['year'] = master['order_date'].dt.year
master['month'] = master['order_date'].dt.month
master['year_month'] = master['order_date'].dt.to_period('M')
master['day_of_week'] = master['order_date'].dt.day_name()
master['days_from_signup'] = (master['order_date'] - master['signup_date']).dt.days

print(f"Master table: {master.shape}")
print(f"\nColumns:\n{list(master.columns)}")
print(f"\nMemory: {master.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Save to /kaggle/working/ so you don't rebuild every session
master.to_parquet('/kaggle/working/master.parquet')
print("\n✅ Saved to /kaggle/working/master.parquet")

In [ ]:
master.head() 

In [ ]:
# ─────────────────────────────────────────────────
# PHASE A1 — Headline revenue trend
# ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

sales = tables['sales'].copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

sales['year'] = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month
sales['year_month'] = sales['Date'].dt.to_period('M')
sales['day_of_week'] = sales['Date'].dt.day_name()
sales['gross_profit'] = sales['Revenue'] - sales['COGS']
sales['gross_margin'] = sales['gross_profit'] / sales['Revenue']

# Summary
print("=" * 60)
print("REVENUE OVERVIEW (2012-07-04 → 2022-12-31)")
print("=" * 60)
print(f"Total revenue:          ${sales['Revenue'].sum():>15,.0f}")
print(f"Total COGS:             ${sales['COGS'].sum():>15,.0f}")
print(f"Total gross profit:     ${sales['gross_profit'].sum():>15,.0f}")
print(f"Overall gross margin:   {sales['gross_profit'].sum() / sales['Revenue'].sum():>15.1%}")
print(f"Days covered:           {len(sales):>15,}")
print(f"Avg daily revenue:      ${sales['Revenue'].mean():>15,.0f}")
print(f"Median daily revenue:   ${sales['Revenue'].median():>15,.0f}")
print(f"Std daily revenue:      ${sales['Revenue'].std():>15,.0f}")
print(f"Max day:                ${sales['Revenue'].max():>15,.0f} on {sales.loc[sales['Revenue'].idxmax(), 'Date'].date()}")
print(f"Min day:                ${sales['Revenue'].min():>15,.0f} on {sales.loc[sales['Revenue'].idxmin(), 'Date'].date()}")

# Yearly breakdown
print("\n" + "=" * 60)
print("YEAR-BY-YEAR")
print("=" * 60)
yearly = sales.groupby('year').agg(
    revenue=('Revenue', 'sum'),
    cogs=('COGS', 'sum'),
    avg_daily=('Revenue', 'mean'),
).round(0)
yearly['gross_profit'] = yearly['revenue'] - yearly['cogs']
yearly['margin_pct'] = (yearly['gross_profit'] / yearly['revenue'] * 100).round(1)
yearly['yoy_growth_pct'] = (yearly['revenue'].pct_change() * 100).round(1)
print(yearly[['revenue', 'avg_daily', 'margin_pct', 'yoy_growth_pct']].to_string())

# Monthly aggregate for charting
monthly = sales.groupby('year_month').agg(
    revenue=('Revenue', 'sum'),
    cogs=('COGS', 'sum'),
).reset_index()
monthly['gross_margin'] = (monthly['revenue'] - monthly['cogs']) / monthly['revenue']
monthly['year_month_ts'] = monthly['year_month'].dt.to_timestamp()

# Dual-panel chart: revenue and margin trend
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(monthly['year_month_ts'], monthly['revenue'] / 1e6, 
             linewidth=1.5, color='#2E86AB')
axes[0].fill_between(monthly['year_month_ts'], monthly['revenue'] / 1e6, 
                      alpha=0.15, color='#2E86AB')
axes[0].set_title('Monthly Revenue and Gross Margin — 2012 to 2022', 
                   fontsize=13, fontweight='bold')
axes[0].set_ylabel('Revenue (millions)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly['year_month_ts'], monthly['gross_margin'] * 100, 
             linewidth=1.5, color='#A23B72')
axes[1].axhline(y=sales['gross_profit'].sum() / sales['Revenue'].sum() * 100, 
                color='gray', linestyle='--', alpha=0.5, label='Overall avg')
axes[1].set_ylabel('Gross Margin (%)')
axes[1].set_xlabel('Month')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()